In [8]:
import os
import random
import string
import math

import hail as hl
from ukb_utils import hail_init
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import numpy as np
from ko_utils import ko
from ko_utils import io
from ko_utils import samples

In [2]:
os.chdir('/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/')
hail_init.hail_bmrc_init('logs/hail/hail_test_export.log', 'GRCh38')

Running on Apache Spark version 3.1.2
SparkUI available at http://compe081.hpc.in.bmrc.ox.ac.uk:4040
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.77-684f32d73643
LOGGING: writing to logs/hail/hail_test_export.log


In [25]:
directory = "data/mt/csqs"
input_path = directory + "/ukb_eur_wes_200k_chr7_maf0to5e-2_pLoF_damaging_missense.mt"
input_type = "mt"

In [26]:
path

'data/mt/csq/ukb_eur_wes_200k_chr17_maf0to5e-2_pLoF_damaging_missense.mt'

In [27]:
mt = io.import_table(input_path, input_type)

In [28]:
expr = mt.consequence.vep.worst_csq_by_gene_canonical.gene_id
mt = (mt.group_rows_by(expr)
                    .aggregate(
                        unphased_het=hl.agg.count_where((mt.GT.is_het_ref()) & (~mt.GT.phased)),
                        phased_het=hl.agg.count_where((mt.GT.is_het_ref()) & (mt.GT.phased)),
                        hom_alt_n=hl.agg.count_where(mt.GT.is_hom_var())
                        )
               )

In [29]:
m = mt.checkpoint("mt_checkpoint.mt", overwrite = True)
mt = m

2022-05-03 15:38:28 Hail: INFO: Ordering unsorted dataset with network shuffle
2022-05-03 15:40:35 Hail: INFO: wrote matrix table with 866 rows and 176915 columns in 866 partitions to mt_checkpoint.mt
    Total size: 4.90 MiB
    * Rows/entries: 3.74 MiB
    * Columns: 1.15 MiB
    * Globals: 11.00 B
    * Smallest partition: 1 rows (2.38 KiB)
    * Largest partition:  1 rows (73.17 KiB)


In [30]:
mt.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'gene_id': str
----------------------------------------
Entry fields:
    'unphased_het': int64
    'phased_het': int64
    'hom_alt_n': int64
----------------------------------------
Column key: ['s']
Row key: ['gene_id']
----------------------------------------


In [31]:
mt = mt.filter_rows(mt.gene_id == "ENSG00000164692")

In [34]:
ht = mt.entries()

2022-05-03 15:53:24 Hail: WARN: entries(): Resulting entries table is sorted by '(row_key, col_key)'.
    To preserve row-major matrix table order, first unkey columns with 'key_cols_by()'


In [41]:
mt.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'gene_id': str
----------------------------------------
Entry fields:
    'unphased_het': int64
    'phased_het': int64
    'hom_alt_n': int64
----------------------------------------
Column key: ['s']
Row key: ['gene_id']
----------------------------------------


In [39]:
ht.filter((ht.phased_het > 1) | (ht.hom_alt_n > 0) ).show()

,,,,
gene_id,s,unphased_het,phased_het,hom_alt_n
str,str,int64,int64,int64
"""ENSG00000164692""","""1326715""",0,2,0
"""ENSG00000164692""","""1890022""",0,2,0
"""ENSG00000164692""","""2301892""",0,2,0
"""ENSG00000164692""","""2314440""",0,2,0
"""ENSG00000164692""","""5571743""",0,2,0
